# CSI4142 Assignment 2
Jonathan Cojita, Lucas Gavura

## Part 2: Imputation
Jonathan Cojita

In [4]:
#Imports needed libraries
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
import numpy as np
from collections import defaultdict

/Users/jonathanc/dev/school/Data_Science/DataScienceA2/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/jonathanc/dev/school/Data_Science/DataScienceA2/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
#Import dataset and preview
path = kagglehub.dataset_download(
    "yashdev01/spotify-tracks-dataset"
)

df = pd.read_csv(f"{path}/spotify-tracks-dataset.csv")
print(df.head())

   Unnamed: 0.1  Unnamed: 0                track_id                 artists  \
0             0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1             1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2             2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   
3             3           3  6lfxq3CG4xtTiEg7opyCyx            Kina Grannis   
4             4           4  5vjLSffimiIP26QG5WcN2K        Chord Overstreet   

                                          album_name  \
0                                             Comedy   
1                                   Ghost (Acoustic)   
2                                     To Begin Again   
3  Crazy Rich Asians (Original Motion Picture Sou...   
4                                            Hold On   

                   track_name  popularity  duration_ms  explicit  \
0                      Comedy          73       230666     False   
1            Ghost - Acoustic          55       1496

### 2.1:
**Type of Imputation**:
**Description of Imputation**:

**Type of Missing Data Simulated**:
**Attrtibute Affected**:
**How Missing Data is Simulated**:


speechiness - MCAR (Median)
Genre - MAR (Similarity Based)
Popularity - MCAR (Regression)


In [19]:
#Simulate MCAR over genre column
missing_rate = 0.1
n_missing = int(len(df) * missing_rate)
missing_indices = np.random.choice(df.index, n_missing, replace=False)
df.loc[missing_indices, 'track_genre'] = np.nan

#Print missing value row sample (Print track, artist, and genre)
print(df[df['track_genre'].isna()].head()[['track_name', 'artists', 'track_genre']])

               track_name                    artists track_genre
1        Ghost - Acoustic               Ben Woodward         NaN
2          To Begin Again     Ingrid Michaelson;ZAYN         NaN
8                   Lucky  Jason Mraz;Colbie Caillat         NaN
10   Give Me Your Forever               Zack Tabudlo         NaN
16  ily (i love you baby)       Andrew Foy;Renee Foy         NaN


In [ ]:
#Calculate the mode of the genre column

genre_counts = defaultdict(int)
median_genre_count = 0

for genre in df['track_genre']:
    if pd.notna(genre):
        genre_counts[genre] += 1

keys = list(genre_counts.keys()).sorted()

if len(keys) % 2 == 0:
    #Take average of the two middle values
    median_genre = (keys[len(keys) // 2 - 1] + keys[len(keys) // 2]) / 2
    median_genre_count = (genre_counts[keys[len(keys) // 2 - 1]] + genre_counts[keys[len(keys) // 2]]) / 2
else:
    median_genre = keys[len(keys) // 2]
    median_genre_count = genre_counts[median_genre]

print(f"Median genre: {median_genre}, Count: {median_genre_count}")

Mode genre: samba, Count: 8926


In [21]:
#Impute missing values with mode
imputed_indices = df[df['track_genre'].isna()].index
df['track_genre'] = df['track_genre'].fillna(mode_genre)

#Print the rows that were just imputed to confirm
print(f"Imputed {len(imputed_indices)} rows")
print(df.loc[imputed_indices, ['track_name', 'artists', 'track_genre']].head())

Imputed 30902 rows
               track_name                    artists track_genre
1        Ghost - Acoustic               Ben Woodward       samba
2          To Begin Again     Ingrid Michaelson;ZAYN       samba
8                   Lucky  Jason Mraz;Colbie Caillat       samba
10   Give Me Your Forever               Zack Tabudlo       samba
16  ily (i love you baby)       Andrew Foy;Renee Foy       samba
